# DentalPlus AI — Desarrollo del agente inteligente

## Etapa 3: Recuperación semántica mediante RAG

In [1]:
import sys
import os
import sqlite3

print("Proyecto: DentalPlus AI")
print("Python:", sys.version)
print("Directorio:", os.getcwd())
print("SQLite disponible correctamente")

Proyecto: DentalPlus AI
Python: 3.14.6 (tags/v3.14.6:c63aec6, Jun 10 2026, 10:26:10) [MSC v.1944 64 bit (AMD64)]
Directorio: c:\Users\fusk5\Desktop\dentalplus-ai\notebooks
SQLite disponible correctamente


#### Importar el módulo

In [2]:
import sys
from pathlib import Path

proyecto = Path.cwd()

if proyecto.name == "notebooks":
    proyecto = proyecto.parent

if str(proyecto) not in sys.path:
    sys.path.insert(0, str(proyecto))

from src.rag import (
    cargar_documentos,
    dividir_documentos,
    crear_vectorstore,
    buscar_informacion,
)

#### Cargar documentos

In [3]:
documentos = cargar_documentos()

print(f"Documentos cargados: {len(documentos)}")

for doc in documentos:
    print(Path(doc.metadata["source"]).name)

Documentos cargados: 4
01_servicios.md
02_profesionales.md
03_politicas_reserva.md
04_preguntas_frecuentes.md


#### Fragmentación

In [4]:
fragmentos = dividir_documentos(documentos)

print(f"Fragmentos generados: {len(fragmentos)}")

print("\nEjemplo de fragmento:")
print(fragmentos[0].page_content)

Fragmentos generados: 14

Ejemplo de fragmento:
# Clínica DentalPlus — Servicios y precios (SIMULADOS)
Versión: 1.0 | Uso: prototipo académico | Fuente institucional ficticia.

La clínica ofrece exclusivamente estos servicios reservables en el prototipo:


#### Crear base vectorial

In [5]:
vectorstore = crear_vectorstore()

print("Base vectorial FAISS creada correctamente.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Base vectorial FAISS creada correctamente.


#### Primera búsqueda semántica

In [6]:
consulta = "¿Cuánto cuesta una limpieza dental?"

resultados = buscar_informacion(
    vectorstore,
    consulta,
    k=3,
)

for i, resultado in enumerate(resultados, start=1):
    print(f"\nResultado {i}")
    print("Fuente:", resultado["fuente"])
    print("Contenido:", resultado["contenido"])


Resultado 1
Fuente: 04_preguntas_frecuentes.md
Contenido: # Clínica DentalPlus — Preguntas frecuentes (SIMULADAS)
Versión: 1.0 | Uso: prototipo académico.

## ¿Qué tratamientos ofrecen?
Evaluación dental, limpieza dental y evaluación de endodoncia. Consultar el documento de servicios para sus precios referenciales.

## ¿Cuánto cuesta una limpieza dental?
El precio simulado es de $35.000 CLP.

## ¿Con quién puedo agendar una limpieza dental?
Con Carlos Rojas, profesional ficticio asociado a higiene dental.

Resultado 2
Fuente: 01_servicios.md
Contenido: La clínica ofrece exclusivamente estos servicios reservables en el prototipo:

| ID | Especialidad | Servicio | Precio referencial CLP | Duración |
|---|---|---|---:|---:|
| 1 | Odontología general | Evaluación dental | $20.000 | 30 minutos |
| 2 | Higiene dental | Limpieza dental | $35.000 | 45 minutos |
| 3 | Endodoncia | Evaluación de endodoncia | $30.000 | 30 minutos |

Resultado 3
Fuente: 01_servicios.md
Contenido: # Clínica Dental

In [7]:
consulta = "¿Cómo puedo solicitar una hora dental?"
resultados = buscar_informacion(vectorstore, consulta, k=3)
for resultado in resultados:
    print(resultado["fuente"])
    print(resultado["contenido"])
    print("---")

04_preguntas_frecuentes.md
## ¿Pueden decirme qué enfermedad dental tengo?
No. El asistente solo ofrece información administrativa y debe derivar consultas clínicas a un profesional.

## ¿Qué medios de pago aceptan?
No existe información autorizada en el prototipo. El asistente debe reconocer esa limitación, sin inventar medios de pago.

## ¿Cómo puedo solicitar una hora dental?

Para solicitar una hora en DentalPlus, el paciente debe
indicar qué servicio necesita.
---
04_preguntas_frecuentes.md
Para solicitar una hora en DentalPlus, el paciente debe
indicar qué servicio necesita.

El asistente consulta la disponibilidad de horarios
en la base de datos de la clínica.

Cuando el paciente selecciona un horario disponible,
el asistente solicita un alias ficticio y una
confirmación explícita antes de registrar la reserva.

La reserva solo se considera confirmada cuando
el sistema la registra correctamente en la base de datos.
---
01_servicios.md
La clínica ofrece exclusivamente estos servi

In [8]:
import pandas as pd

consultas = [
    "¿Cuánto cuesta una limpieza dental?",
    "¿Cómo puedo solicitar una hora dental?",
    "¿Quién realiza las limpiezas dentales?",
    "¿Cuál es la política de cancelación?"
]

filas = []

for consulta in consultas:
    resultados = buscar_informacion(
        vectorstore,
        consulta,
        k=3,
        umbral=0.50
    )

    if not resultados:
        filas.append({
            "Consulta": consulta,
            "Fuente": "Sin resultados",
            "Similitud": None,
            "Fragmento": ""
        })

    for resultado in resultados:
        filas.append({
            "Consulta": consulta,
            "Fuente": resultado["fuente"],
            "Similitud": resultado["similitud"],
            "Fragmento": resultado["contenido"][:100]
        })

df_resultados = pd.DataFrame(filas)
display(df_resultados)

,Consulta,Fuente,Similitud,Fragmento
0,¿Cuánto cuesta una limpieza dental?,04_preguntas_frecuentes.md,0.701,# Clínica DentalPlus — Preguntas frecuentes (S...
1,¿Cuánto cuesta una limpieza dental?,01_servicios.md,0.593,La clínica ofrece exclusivamente estos servici...
2,¿Cuánto cuesta una limpieza dental?,01_servicios.md,0.526,# Clínica DentalPlus — Servicios y precios (SI...
3,¿Cómo puedo solicitar una hora dental?,04_preguntas_frecuentes.md,0.692,## ¿Pueden decirme qué enfermedad dental tengo...
4,¿Cómo puedo solicitar una hora dental?,04_preguntas_frecuentes.md,0.675,"Para solicitar una hora en DentalPlus, el paci..."
5,¿Cómo puedo solicitar una hora dental?,01_servicios.md,0.592,La clínica ofrece exclusivamente estos servici...
6,¿Quién realiza las limpiezas dentales?,02_profesionales.md,0.663,# Clínica DentalPlus — Profesionales (SIMULADO...
7,¿Quién realiza las limpiezas dentales?,04_preguntas_frecuentes.md,0.586,# Clínica DentalPlus — Preguntas frecuentes (S...
8,¿Quién realiza las limpiezas dentales?,01_servicios.md,0.580,# Clínica DentalPlus — Servicios y precios (SI...
9,¿Cuál es la política de cancelación?,03_politicas_reserva.md,0.521,"5. Si la hora ya no está disponible, el asiste..."


# LLM


In [9]:
import os
from pathlib import Path
from dotenv import load_dotenv

raiz = Path.cwd()

if raiz.name == "notebooks":
    raiz = raiz.parent

load_dotenv(raiz / ".env")

print(
    "API Key configurada:",
    bool(os.getenv("GOOGLE_API_KEY"))
)

API Key configurada: True


In [10]:
from src.llm import crear_llm

llm = crear_llm()

respuesta = llm.invoke(
    "Responde únicamente con la palabra: Conectado"
)

print(respuesta.text())

c:\Users\fusk5\Desktop\dentalplus-ai\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Conectado


c:\Users\fusk5\Desktop\dentalplus-ai\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: Calling .text() as a method is deprecated. Use .text as a property instead (e.g., message.text).
  exec(code_obj, self.user_global_ns, self.user_ns)


##  — Generación de respuestas con RAG y Gemini

In [11]:
from src.asistente_rag import responder_con_rag

resultado = responder_con_rag(
    pregunta="¿Cuánto cuesta una limpieza dental?",
    vectorstore=vectorstore,
    llm=llm,
)

print("RESPUESTA:")
print(resultado["respuesta"])

print("\nFUENTES:")
for fuente in resultado["fuentes"]:
    print("-", fuente)

c:\Users\fusk5\Desktop\dentalplus-ai\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


RESPUESTA:
El precio simulado de una limpieza dental es de $35.000 CLP.

FUENTES:
- 04_preguntas_frecuentes.md
- 01_servicios.md


## Probar preguntas conocidas y desconocidas

In [12]:
preguntas = [
    "¿Qué tratamientos ofrece DentalPlus?",
    "¿Quién realiza las limpiezas dentales?",
    "¿Cómo puedo reservar una hora?",
    "¿Puedo cancelar una reserva?",
    "¿Aceptan tarjetas de crédito?",
    "¿Tengo una caries si me duele una muela?",
]

for pregunta in preguntas:
    resultado = responder_con_rag(
        pregunta=pregunta,
        vectorstore=vectorstore,
        llm=llm,
    )

    print("\nPREGUNTA:", pregunta)
    print("RESPUESTA:", resultado["respuesta"])
    print("FUENTES:", resultado["fuentes"])
    print("-" * 60)

c:\Users\fusk5\Desktop\dentalplus-ai\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



PREGUNTA: ¿Qué tratamientos ofrece DentalPlus?
RESPUESTA: ¡Hola! Con gusto te informo. Según nuestros registros, la clínica ofrece exclusivamente los siguientes tratamientos reservables: evaluación dental, limpieza dental y evaluación de endodoncia. 

Si necesitas conocer los precios referenciales, puedes consultar el documento de servicios de la clínica. ¡Que tengas un excelente día!
FUENTES: ['03_politicas_reserva.md', '01_servicios.md', '04_preguntas_frecuentes.md']
------------------------------------------------------------


c:\Users\fusk5\Desktop\dentalplus-ai\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



PREGUNTA: ¿Quién realiza las limpiezas dentales?
RESPUESTA: Hola. Las limpiezas dentales las realiza Carlos Rojas, profesional ficticio asociado a higiene dental.
FUENTES: ['02_profesionales.md', '04_preguntas_frecuentes.md', '01_servicios.md']
------------------------------------------------------------


c:\Users\fusk5\Desktop\dentalplus-ai\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



PREGUNTA: ¿Cómo puedo reservar una hora?
RESPUESTA: ¡Hola! Para solicitar una reserva, debes elegir un servicio y una hora ofrecida por la agenda. Luego, se solicitará un alias ficticio para identificar la reserva de demostración y se pedirá tu confirmación explícita antes del registro. Recuerda que no debes entregar información personal real.
FUENTES: ['04_preguntas_frecuentes.md', '03_politicas_reserva.md']
------------------------------------------------------------


c:\Users\fusk5\Desktop\dentalplus-ai\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



PREGUNTA: ¿Puedo cancelar una reserva?
RESPUESTA: El prototipo no ejecuta cancelaciones ni cambios de reserva. Para realizar este tipo de solicitudes, debes derivar a recepción simulada y recordar que no se pueden hacer cambios automáticos.
FUENTES: ['04_preguntas_frecuentes.md', '03_politicas_reserva.md']
------------------------------------------------------------

PREGUNTA: ¿Aceptan tarjetas de crédito?
RESPUESTA: No dispongo de información autorizada para responder esa consulta.
FUENTES: []
------------------------------------------------------------

PREGUNTA: ¿Tengo una caries si me duele una muela?
RESPUESTA: No dispongo de información autorizada para responder esa consulta.
FUENTES: []
------------------------------------------------------------


In [15]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel

llm_prueba = FakeListChatModel(
    responses=[
        "Esta es una respuesta simulada de DentalPlus."
    ]
)

resultado = responder_con_rag(
    pregunta="¿Cuánto cuesta una limpieza dental?",
    vectorstore=vectorstore,
    llm=llm_prueba,
    umbral=0.30
)

print(resultado["respuesta"])
print(resultado["fuentes"])

Esta es una respuesta simulada de DentalPlus.
['04_preguntas_frecuentes.md', '01_servicios.md']
